# Notebook 07: Interpretación de Casos de Estudio

Traducción de los clusters de riesgo a conclusiones estratégicas y recomendaciones de negocio.

Este notebook cierra el pipeline de GeoRisk Finder. Toma los resultados del clustering (K-Means + DBSCAN) y los interpreta en términos accionables para tomadores de decisión.

**Datos de entrada:**
- `data/processed/interpretacion_clusters.csv` — Traducción de cada cluster a nombre comercial y recomendación
- `data/processed/casos_estudio.csv` — Análisis por país de la distribución de clusters
- `data/processed/grid_features.csv` — Features originales por celda H3

## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA = Path('../data/processed')

clusters = pd.read_csv(DATA / 'interpretacion_clusters.csv')
casos = pd.read_csv(DATA / 'casos_estudio.csv')
grid = pd.read_csv(DATA / 'grid_features.csv')

print(f'Clusters: {len(clusters)} filas')
print(f'Casos de estudio: {len(casos)} países')
print(f'Celdas H3: {len(grid)}')

## 2. Interpretación de clusters

Cada cluster se traduce a un nombre comercial y una recomendación de negocio.

In [ ]:
clusters_display = clusters[['cluster', 'n_celdas', 'pct_celdas', 'nivel_sismico', 'nivel_ciclonico', 'nivel_volcanico', 'nombre_negocio', 'recomendacion']].copy()
clusters_display['pct_celdas'] = clusters_display['pct_celdas'].round(1)
clusters_display

### Visualización: distribución de clusters

In [ ]:
fig = px.bar(
    clusters.sort_values('cluster'),
    x='cluster',
    y='pct_celdas',
    color='nombre_negocio',
    title='Distribución de celdas por cluster de riesgo',
    labels={'pct_celdas': '% de celdas', 'cluster': 'Cluster'},
    text='pct_celdas'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='Cluster')
fig.show()

### Perfil de riesgo por cluster

Visualización de los tres niveles de riesgo (sísmico, cíclico, volcánico) para cada cluster.

In [ ]:
risk_levels = ['nivel_sismico', 'nivel_ciclonico', 'nivel_volcanico']
level_map = {'Bajo': 1, 'Medio': 2, 'Alto': 3}

fig = go.Figure()
for _, row in clusters.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[level_map[row[c]] for c in risk_levels],
        theta=['Sísmico', 'Cíclico', 'Volcánico'],
        fill='toself',
        name=f'Cluster {row["cluster"]} - {row["nombre_negocio"]}'
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 3])),
    title='Perfil de riesgo por cluster',
    showlegend=True
)
fig.show()

## 3. Casos de estudio por país

Análisis de cómo se distribuyen los clusters en los países con mayor exposición.

In [ ]:
casos_display = casos.sort_values('n_celdas', ascending=False)
casos_display

In [ ]:
fig = px.bar(
    casos.sort_values('n_celdas', ascending=True),
    x='n_celdas',
    y='pais',
    color='nombre_negocio_dominante',
    title='Celdas H3 por país (cluster dominante resaltado)',
    labels={'n_celdas': 'Número de celdas H3', 'pais': 'País', 'nombre_negocio_dominante': 'Cluster dominante'},
    orientation='h'
)
fig.update_layout(showlegend=True)
fig.show()

## 4. Recomendaciones estratégicas

Resumen de las recomendaciones de negocio por cluster.

In [ ]:
for _, row in clusters.iterrows():
    print(f'\n--- Cluster {row["cluster"]}: {row["nombre_negocio"]} ---')
    print(f'  Celdas: {row["n_celdas"]} ({row["pct_celdas"]:.1f}%)')
    print(f'  Sísmico: {row["nivel_sismico"]} | Cíclico: {row["nivel_ciclonico"]} | Volcánico: {row["nivel_volcanico"]}')
    print(f'  Recomendación: {row["recomendacion"]}')

## 5. Mapa de calor global de riesgo

Visualización geoespacial de las celdas H3 coloreadas por cluster.

In [ ]:
import plotly.express as px

fig = px.density_mapbox(
    grid,
    lat='lat',
    lon='lon',
    z='eq_count',
    radius=3,
    center=dict(lat=20, lon=0),
    zoom=1,
    mapbox_style='carto-positron',
    title='Densidad de eventos sísmicos global (celdas H3)',
    color_continuous_scale='Reds'
)
fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

## 6. Conclusiones

1. **Cluster 1 (67.2%)**: Riesgo sísmico bajo — mayoría de celdas. Buen candidato para desarrollo inmobiliario sin refuerzos especiales.
2. **Cluster 0 (10.1%)**: Riesgo cíclico alto — exigir normativa anti-huracán y primas ajustadas a temporada.
3. **Cluster 3 (6.3%)**: Riesgo sísmico alto — exigir normativa antisísmica.
4. **Cluster 4 (11.8%)**: Riesgo volcánico alto — monitoreo periódico de actividad volcánica.
5. **Cluster 2 (4.6%)**: Riesgo alto en las tres dimensiones — cobertura especializada y reaseguro.

El análisis de casos de estudio confirma que Japón y Chile concentran el mayor riesgo sísmico, mientras que el sureste asiático presenta mayor exposición cíclica.